<a href="https://colab.research.google.com/github/naman-0804/learning/blob/langgraph/Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agentic RAG**

In [2]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [23]:
# Install necessary libraries
!pip install -q langchain langchain-community langchain-google-genai
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 104.5 MB/s eta 0:00:00


In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [19]:
urls=[
    "https://docs.langchain.com/oss/python/langgraph/overview",
    "https://docs.langchain.com/oss/python/langgraph/workflows-agents",
    "https://docs.langchain.com/oss/python/langgraph/graph-api#map-reduce-and-the-send-api"
]
docs=[WebBaseLoader(urls).load() for url in urls]
docs

[[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview', 'title': 'LangGraph overview - Docs by LangChain', 'description': 'Gain control with LangGraph to design agents that reliably handle complex tasks', 'language': 'en'}, page_content='LangGraph overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraph overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartLocal serverChangelogThinking in LangGraphWorkflows + agentsCapabilitiesPersistenceCheckpointersStoresFault toleranceEvent stre

In [20]:
doc_list=[item for sublist in docs for item in sublist]
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
doc_splits=text_splitter.split_documents(doc_list)
print(doc_splits[:2])

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview', 'title': 'LangGraph overview - Docs by LangChain', 'description': 'Gain control with LangGraph to design agents that reliably handle complex tasks', 'language': 'en'}, page_content="LangGraph overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraph overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartLocal serverChangelogThinking in LangGraphWorkflows + agentsCapabilitiesPersistenceCheckpointersStoresFault toleranceEvent stream

In [28]:
vectorstore=FAISS.from_documents(
    documents=doc_splits,
    embedding=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2',api_key=GOOGLE_API_KEY)
)
retriever=vectorstore.as_retriever()

In [30]:
retriever.invoke("What is langgraph?")

[Document(id='918f65b1-5fa8-493a-9823-8f6090bee6ee', metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview', 'title': 'LangGraph overview - Docs by LangChain', 'description': 'Gain control with LangGraph to design agents that reliably handle complex tasks', 'language': 'en'}, page_content='Then, create a simple hello world example:\nfrom langgraph.graph import StateGraph, MessagesState, START, END\n\ndef mock_llm(state: MessagesState):\n    return {"messages": [{"role": "ai", "content": "hello world"}]}\n\ngraph = StateGraph(MessagesState)\ngraph.add_node(mock_llm)\ngraph.add_edge(START, "mock_llm")\ngraph.add_edge("mock_llm", END)\ngraph = graph.compile()\n\ngraph.invoke({"messages": [{"role": "user", "content": "hi!"}]})\n\nUse LangSmith to trace requests, debug agent behavior, and evaluate outputs. Set LANGSMITH_TRACING=true and your API key to get started. Follow the tracing quickstart to get set up.  We recommend you also set up LangSmith Engine which monit

In [35]:
#Retriever to Retriever Tools
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(
    retriever,
    "retriver_vector_Db_blog",
    "Search and run information about Langraph"
)

In [36]:
retriever_tool

StructuredTool(name='retriver_vector_Db_blog', description='Search and run information about Langraph', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7fa227b55b20>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7fa227b542c0>)

In [37]:
##Langchain blogs creating sepearte dbs to see how they work
langchain_urls=[
    "https://docs.langchain.com/oss/python/langchain/overview",
    "https://docs.langchain.com/oss/python/langchain/models"
]
docs=[WebBaseLoader(urls).load() for url in urls]
docs

[[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview', 'title': 'LangGraph overview - Docs by LangChain', 'description': 'Gain control with LangGraph to design agents that reliably handle complex tasks', 'language': 'en'}, page_content='LangGraph overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraph overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartLocal serverChangelogThinking in LangGraphWorkflows + agentsCapabilitiesPersistenceCheckpointersStoresFault toleranceEvent stre

In [38]:
doc_list=[item for sublist in docs for item in sublist]
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=100)
doc_splits=text_splitter.split_documents(doc_list)
print(doc_splits[:2])

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langgraph/overview', 'title': 'LangGraph overview - Docs by LangChain', 'description': 'Gain control with LangGraph to design agents that reliably handle complex tasks', 'language': 'en'}, page_content="LangGraph overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangGraph overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartLocal serverChangelogThinking in LangGraphWorkflows + agentsCapabilitiesPersistenceCheckpointersStoresFault toleranceEvent stream

In [39]:
vectorstorelangchain=FAISS.from_documents(
    documents=doc_splits,
    embedding=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2',api_key=GOOGLE_API_KEY)
)
retrieverlangchain=vectorstore.as_retriever()

In [40]:
#Retriever to Retriever Tools
from langchain_core.tools import create_retriever_tool
retriever_tool=create_retriever_tool(
    retriever,
    "retriver_vector_langchain_blog",
    "Search and run information about Langchain"
)

In [41]:
tools=[retriever_tool,retrieverlangchain]

In [ ]:
#Langraph workflow

In [42]:
#!pip install -q langgraph

In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Define the state
class State(TypedDict):
    # Use `add_messages` to properly append tool outputs to the conversation history
    messages: Annotated[list, add_messages]

# 2. Define the node (the logic)
def call_model(state: State):
    # Bind the tools to the LLM so it knows it can search your vectors
    llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", api_key=GOOGLE_API_KEY).bind_tools(tools)
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

# 3. Create the graph
workflow = StateGraph(State)

# Add the standard agent node and a specific node to execute tools
workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))

# Set the entrypoint
workflow.add_edge(START, "agent")

# Add conditional edge: if the agent calls a tool, route to "tools" node, otherwise route to END
workflow.add_conditional_edges("agent", tools_condition)

# After tools run, route back to the agent to interpret the results
workflow.add_edge("tools", "agent")

# 4. Compile
app = workflow.compile()


In [ ]:
# 5. Run the workflow
# Let's test it by asking a question that requires searching the vector database
inputs = {"messages": [("user", "Search the blog and explain what LangGraph is.")]}
result = app.invoke(inputs)

print(result["messages"][-1].content)


[{'type': 'text', 'text': '**LangChain** is an open-source framework that simplifies building AI applications by connecting large language models (LLMs) to external data sources, tools, and APIs.', 'extras': {'signature': 'EroOCrcOARFNMg/MRsp1An1v5fmPpjH1ys95XWlTv8qiHYJG3mEPSh+Oa1XMBmTHVJUeXSQvQpV6DvtHTM00X1vCpbQqqD6hRk4aU2cl+ZEfMVs2ZsaDcQcfQqb3omhuGWSjTZTlVNg2jFG4IGCzgqZtWDSvUtrQPJgpwu0Sjvk5dbKNGXsDrLTf+yBFHzVBdgXgLsU7mfFW3/sAX5m8OfG5zV4s9YvuD0xkXECDQaAphk63AGCkPEKtLD0ks7X6H/zrDXUJwbjwWDD130ULAWvb+OB/AiG57swvCZcGKkIH19CY87A4iZoR6DHn8eMxvCQbF3zLujar9nThIa1hcvAl/PwpIDBP10tORg2hD+b0xs1jOLQ15iurTzWMIOpyKgi7iCNkMt/zQXa5QALKDQt9rphFsJ4iTWhXIJVJwLSFFrjnaGOWN8H0yBlUECcXwMMYOI269oD0hjz/qN/LOdmIBHAZ2LzIVo9rNTWwXVx0luEQCAvx0V/a0LchwrgWxJlhlPqKsHcS4lIF5hmJkzigZkXsapT2bXANbG4HWnio6/aS1MvTNwdCS/b/Ul/Dg3qA5Qs+JQM5jD+27JfQ4+UuyCOEp/SMoCsR1clEDt7H0cbBRojclvXSdUvBs/fiXoOReVxihKi9+j42nNwIXOr/6tZcTg3+YcrknXgcCxc5stNLeDmIE5GwR2Zr1atftR8E2Xvh5tw6BgRzYru0JocZXkZx3MAR5PJpVuKhQ618VSA2o5XUNQlLAUDmNBlN1mOF/TGw1